In [0]:
%python
bronze_path = '/Volumes/workspace/techvenda/filestore/bronze/'
silver_path = '/Volumes/workspace/techvenda/filestore/silver/'
gold_path = '/Volumes/workspace/techvenda/filestore/gold/'
origem_path = '/Volumes/workspace/techvenda/filestore/origem/'

In [0]:
%python
#Tabelas temporarias
bronze_mapeamento= {
    'temp_bronze_clientes' : f'{bronze_path}/clientes/',
    'temp_bronze_itens_pedido' : f'{bronze_path}/itens_pedido/',
    'temp_bronze_pedidos' : f'{bronze_path}/pedidos/',
    'temp_bronze_produtos' : f'{bronze_path}/produtos/',
    'temp_bronze_vendedores' : f'{bronze_path}/vendedores/'

}
for view_name, path in bronze_mapeamento.items():
    (spark.read.format('delta')
        .load(path)
        .createOrReplaceTempView(view_name)
    )

In [0]:
%sql
select * from temp_bronze_vendedores

In [0]:
%sql
describe temp_bronze_vendedores

In [0]:
%python
df_vendedores = spark.sql("""
    SELECT
        id_vendedor,
        nome_vendedor,
        estado,
        regiao,
        salario_base,
        DATE_FORMAT(data_contratacao, 'dd/MM/yyyy') as data_contratacao,
        status
        

    FROM temp_bronze_vendedores

    WHERE LOWER(TRIM(status)) <> 'inativo'

    GROUP BY
        id_vendedor, nome_vendedor, estado, regiao, salario_base, data_contratacao, status
""")

# Salvar em delta na silver
df_vendedores.write\
    .mode('overwrite')\
        .format('delta')\
            .option('mergeSchema', 'true')\
                .save(f'{silver_path}/vendedores')

In [0]:
%python
display(df_vendedores)